In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import sys
from pathlib import Path
import json
from scipy.stats import linregress
import data

In [ ]:
ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]

VALIDATION_MONTHS = pd.period_range("2025-12", "2026-05", freq="M")

# Analyze the current modular validation output.
PREDICTIONS_PATH = ROOT / "outputs" / "validation_predictions.csv"
predictions = pd.read_csv(PREDICTIONS_PATH)

listings = data.load("listings")

geo_cols = ["ml_num", "latitude", "longitude", "date_sold", "date_listed"]

plot_df = predictions.merge(
    listings[geo_cols],
    on="ml_num",
    how="left"
)

plot_df["percent_error"] = 100 * (
    plot_df["predicted_price"] - plot_df["sold_price"]
) / plot_df["sold_price"]

plot_df["absolute_percent_error"] = plot_df["percent_error"].abs()

In [ ]:
def bootstrap_ci(values, statistic, repeats=1000):
    """95% interval: recompute the statistic on 1,000 resamples of the homes
    (drawn with replacement) and keep the middle 95% of the results."""
    rng = np.random.default_rng(0)
    results = [statistic(rng.choice(values, size=len(values))) for _ in range(repeats)]
    return [round(float(x), 2) for x in np.percentile(results, [2.5, 97.5])]


def metrics(predictions):
    """Scores for one track. Errors are in percent of the sold price."""
    signed = 100 * (predictions["predicted_price"] - predictions["sold_price"]) / predictions["sold_price"]
    absolute = signed.abs().to_numpy()
    return {
        "n": len(predictions),
        "mape": round(float(np.mean(absolute)), 2),          # mean absolute % error
        "mape_95ci": bootstrap_ci(absolute, np.mean),
        "mdape": round(float(np.median(absolute)), 2),       # median absolute % error
        "mdape_95ci": bootstrap_ci(absolute, np.median),
        "median_signed_error": round(float(np.median(signed)), 2),  # above 0: predicts too high
    }


def graph_n_intervals(df, x, y, n=10, y_label=None):
    """
    Plot the mean and median of y over n quantile intervals of x.

    x, y: column names or pandas Series
    n: number of intervals
    """

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    data = df.copy()
    data["_x"] = data[x] if isinstance(x, str) else x
    data["_y"] = data[y] if isinstance(y, str) else y

    for ax, track in zip(axes, ["off_market", "list_aware"]):
        subset = data[data["track"] == track].copy()
        subset = subset.dropna(subset=["_x", "_y"])

        subset["interval"] = pd.qcut(
            subset["_x"],
            q=n,
            labels=False,
            duplicates="drop"
        )

        grouped = subset.groupby("interval")

        x_values = grouped["_x"].median()
        mean_y = grouped["_y"].mean()
        median_y = grouped["_y"].median()

        ax.plot(x_values, mean_y, marker="o", label="Mean")
        ax.plot(x_values, median_y, marker="s", label="Median")

        ax.axhline(0, color="black", linestyle="--", alpha=0.4)

        ax.set_title(track.replace("_", " ").title())
        ax.set_xlabel(x if isinstance(x, str) else "X")
        ax.grid(alpha=0.3)
        ax.legend()

    axes[0].set_ylabel(y_label or (y if isinstance(y, str) else "Y"))
    fig.tight_layout()
    plt.show()

    return fig, axes


In [ ]:
predictions = pd.read_csv(PREDICTIONS_PATH)

percent_error = 100 * (
    predictions["predicted_price"] - predictions["sold_price"]
) / predictions["sold_price"]

absolute_error = percent_error.abs()

In [ ]:
graph_n_intervals(
    predictions,
    x="sold_price",
    y=absolute_error,
    n=10,
    y_label="Absolute Percentage Error (%)"
)

In [ ]:
graph_n_intervals(
    predictions,
    x="sold_price",
    y=percent_error,
    n=10,
    y_label="Signed Percentage Error (%)"
)

In [ ]:
graph_n_intervals(
    predictions,
    x="predicted_price",
    y=percent_error,
    n=10,
    y_label="Signed Percentage Error (%)"
)

In [ ]:
for track in ["off_market", "list_aware"]:
    df = predictions[predictions["track"] == track]

    log_pred = np.log(df["predicted_price"])
    log_actual = np.log(df["sold_price"])

    result = linregress(log_pred, log_actual)

    print(f"{track}: calibration slope = {result.slope:.4f}")

In [ ]:
def geographic_heatmap(
    df,
    value_col,
    title,
    track=None,
    gridsize=50,
    cmap="viridis",
    vmin=None,
    vmax=10_000_000
):
    """
    Plot a geographic heat map over latitude/longitude using hexbin.

    value_col : column to aggregate in each hex
    track     : "off_market", "list_aware", or None for all
    vmin/vmax : color scale limits
    """
    data = df.copy()

    if track is not None:
        data = data[data["track"] == track].copy()

    data = data.dropna(subset=["longitude", "latitude", value_col])

    plt.figure(figsize=(8, 7))

    hb = plt.hexbin(
        data["longitude"],
        data["latitude"],
        C=data[value_col],
        reduce_C_function=np.mean,
        gridsize=gridsize,
        cmap=cmap,
        mincnt=1,
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(hb, label=value_col.replace("_", " ").title())
    if value_col in ["predicted_price", "sold_price", "list_price"]:
        cbar.ax.set_ylabel("Price ($)")

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title if track is None else f"{title} ({track})")
    plt.tight_layout()
    plt.show()

In [ ]:
geographic_heatmap(
    plot_df,
    value_col="predicted_price",
    title="Predicted House Prices Heat Map",
    track="off_market",
    cmap="viridis",
    vmin=0,
    vmax=8_000_000
)

In [ ]:

geographic_heatmap(
    plot_df,
    value_col="sold_price",
    title="Actual Sold Prices Heat Map",
    track="off_market",
    cmap="viridis",
    vmin=0,
    vmax=8_000_000
)

In [ ]:
geographic_heatmap(
    plot_df,
    value_col="percent_error",
    title="Signed Percent Error Heat Map",
    track="off_market",
    cmap="coolwarm",
    vmin=-80,
    vmax=80
)

In [ ]:
def percent_error_over_time(df, track=None, date_col="month"):
    data = df.copy()

    if track is not None:
        data = data[data["track"] == track].copy()

    if date_col != "month":
        data["month"] = pd.to_datetime(data[date_col]).dt.to_period("M").astype(str)

    data["percent_error"] = 100 * (
        data["predicted_price"] - data["sold_price"]
    ) / data["sold_price"]

    data["absolute_percent_error"] = data["percent_error"].abs()

    grouped = data.groupby("month", observed=True).agg(
        mean_percent_error=("percent_error", "mean"),
        median_percent_error=("percent_error", "median"),
        mape=("absolute_percent_error", "mean"),
        mdape=("absolute_percent_error", "median"),
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(grouped.index, grouped["mean_percent_error"], marker="o", label="Mean % Error")
    ax.plot(grouped.index, grouped["median_percent_error"], marker="s", label="Median % Error")
    ax.axhline(0, color="black", linestyle="--", alpha=0.6)

    ax.set_title("Percent Error Over Time" if track is None else f"Percent Error Over Time ({track})")
    ax.set_xlabel("Month")
    ax.set_ylabel("Signed Percentage Error (%)")
    ax.grid(alpha=0.3)
    ax.legend()

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return grouped

def price_error_over_time(predictions, statistic="median"):
    """
    Plot actual vs predicted prices and signed percentage
    error over time, separately for each track.

    statistic: "median" or "mean" for price aggregation.
    """

    fig, axes = plt.subplots(
        2, 2,
        figsize=(14, 9),
        sharex="col",
        gridspec_kw={"height_ratios": [2, 1]}
    )

    for i, track in enumerate(["off_market", "list_aware"]):
        df = predictions[predictions["track"] == track].copy()

        df["month"] = pd.to_datetime(df["month"])

        df["percent_error"] = 100 * (
            df["predicted_price"] - df["sold_price"]
        ) / df["sold_price"]

        monthly = df.groupby("month").agg(
            actual_price=("sold_price", statistic),
            predicted_price=("predicted_price", statistic),
            mean_error=("percent_error", "mean"),
            median_error=("percent_error", "median")
        ).sort_index()

        # Top: actual and predicted selling prices
        ax = axes[0, i]

        ax.plot(
            monthly.index,
            monthly["actual_price"],
            marker="o",
            label="Actual Price"
        )

        ax.plot(
            monthly.index,
            monthly["predicted_price"],
            marker="s",
            label="Predicted Price"
        )

        ax.set_title(track.replace("_", " ").title())
        ax.set_ylabel(f"{statistic.title()} Selling Price ($)")
        ax.yaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f"${x / 1e6:.2f}M")
        )
        ax.grid(alpha=0.3)
        ax.legend()

        # Bottom: signed percentage error
        ax = axes[1, i]

        ax.plot(
            monthly.index,
            monthly["mean_error"],
            marker="o",
            label="Mean % Error"
        )

        ax.plot(
            monthly.index,
            monthly["median_error"],
            marker="s",
            label="Median % Error"
        )

        ax.axhline(0, color="black", linestyle="--", alpha=0.6)

        ax.set_ylabel("Signed Percentage Error (%)")
        ax.set_xlabel("Sale Month")
        ax.grid(alpha=0.3)
        ax.legend()

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%b %Y")
        )
        ax.tick_params(axis="x", rotation=45)

    fig.suptitle("Model Predictions vs Actual Prices Over Time", fontsize=15)
    fig.tight_layout()

    plt.show()
    return fig, axes

In [ ]:
percent_error_over_time(plot_df, track="off_market")
percent_error_over_time(plot_df, track="list_aware")

price_error_over_time(predictions, statistic="median")
price_error_over_time(predictions, statistic="mean")

In [ ]:
df = predictions.copy()

df["signed_error"] = 100 * (
    df["predicted_price"] - df["sold_price"]
) / df["sold_price"]

df["absolute_error"] = df["signed_error"].abs()

monthly = df.groupby(["track", "month"]).agg(
    n=("sold_price", "size"),
    MAPE=("absolute_error", "mean"),
    MdAPE=("absolute_error", "median"),
    Mean_Bias=("signed_error", "mean"),
    Median_Bias=("signed_error", "median")
).round(2)

display(monthly)

In [ ]:
predictions["signed_error"] = 100 * (
    predictions["predicted_price"] - predictions["sold_price"]
) / predictions["sold_price"]

monthly_type = predictions.groupby(
    ["track", "month", "house_type_name"]
).agg(
    n=("signed_error", "size"),
    mean_bias=("signed_error", "mean"),
    median_bias=("signed_error", "median")
)

display(monthly_type.round(2))

In [ ]:
df = predictions.copy()

df["signed_error"] = 100 * (
    df["predicted_price"] - df["sold_price"]
) / df["sold_price"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, track in zip(axes, ["off_market", "list_aware"]):
    subset = df[df["track"] == track]

    grouped = subset.groupby(
        ["house_type_name", "month"]
    )["signed_error"]

    bias = grouped.mean().unstack()
    counts = grouped.size().unstack()

    # Hide groups with fewer than 30 observations
    bias = bias.where(counts >= 30)

    im = ax.imshow(
        bias,
        cmap="coolwarm",
        norm=TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10),
        aspect="auto"
    )

    ax.set_xticks(range(len(bias.columns)), bias.columns, rotation=45)
    ax.set_yticks(range(len(bias.index)), bias.index)
    ax.set_title(track.replace("_", " ").title())

    for i in range(len(bias.index)):
        for j in range(len(bias.columns)):
            value = bias.iloc[i, j]
            if pd.notna(value):
                ax.text(j, i, f"{value:+.1f}",
                        ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes, label="Mean Signed Error (%)", shrink=0.8)
plt.show()